# 🚦 Gridlock Hackathon 2025 — Traffic Demand Prediction

---

## Overview

This notebook documents the end-to-end pipeline for predicting normalised traffic demand across geospatial zones, built for the **Gridlock 2025 Hackathon**.

> **Target:** `demand` — a continuous value in **[0, 1]** representing normalised traffic load.  
> **Metric:** Root Mean Squared Error (RMSE). Lower is better.

---

## Dataset

| Split | Rows | Columns |
|-------|------|---------|
| Train | 77,299 | 11 |
| Test  | 41,778 | 10 |

**Key features:** `geohash`, `day`, `timestamp`, `RoadType`, `NumberofLanes`, `LargeVehicles`, `Landmarks`, `Temperature`, `Weather`

---

## Pipeline Summary

| Step | Description |
|------|-------------|
| 1 | Exploratory Data Analysis |
| 2 | Missing Value Treatment |
| 3 | Temporal Feature Engineering |
| 4 | Geohash-Based Aggregations |
| 5 | Early Day-49 Signal Extraction |
| 6 | CatBoost Modelling (V1 → V2 → V3) |
| 7 | LightGBM Exploration & Ensemble |
| 8 | Hyperparameter Optimisation |
| 9 | Final Submission Generation |

---

## Results Summary

| Model | Validation Strategy | Time RMSE |
|-------|---------------------|-----------|
| V1 (baseline) | Random 80/20 split | 0.03082 |
| V1 | Time-based (Day 48 → 49) | 0.06468 |
| V2 (+ early Day-49 features) | Time-based | 0.06375 ✅ |
| V3 (+ geo×time features) | Time-based | 0.06613 |
| **Final (Model D, tuned)** | **Full train** | **—** |


---

## 1. Setup

### 1.1 Library Imports

Core libraries: `pandas` and `numpy` for data manipulation; `catboost` and `sklearn` for modelling and evaluation.


In [27]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

### 1.2 Load Data

Load train and test CSVs from disk and inspect shapes.


In [28]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


---

## 2. Data Preprocessing

### 2.1 Missing Value Imputation

Categorical columns (`RoadType`, `Weather`) are filled with the sentinel string `"Missing"` to preserve them as valid category levels for CatBoost.  
The numeric column `Temperature` is imputed using the **train-set median** to avoid data leakage into the test set.


In [29]:
train["RoadType"] = train["RoadType"].fillna("Missing")
test["RoadType"] = test["RoadType"].fillna("Missing")

train["Weather"] = train["Weather"].fillna("Missing")
test["Weather"] = test["Weather"].fillna("Missing")

train["Temperature"] = train["Temperature"].fillna(
    train["Temperature"].median()
)

test["Temperature"] = test["Temperature"].fillna(
    train["Temperature"].median()
)

### 2.2 Timestamp → Minutes Conversion

The `timestamp` column is in `HH:MM` string format. It is converted to **total minutes since midnight** (an integer in [0, 1425]) to produce a continuous numeric representation of time-of-day.


In [30]:
def convert_time(x):
    h, m = map(int, x.split(":"))
    return h * 60 + m

train["time_minutes"] = train["timestamp"].apply(convert_time)
test["time_minutes"] = test["timestamp"].apply(convert_time)

---

## 3. Feature Engineering

### 3.1 Geohash-Based Statistical Features

For each `geohash` zone, aggregate historical demand statistics (`mean`, `std`, `max`, `min`) from the training set and join them back to both train and test.  
These features encode the **long-run demand profile** of each location.

> **Data-leakage note:** These statistics are computed on the full training set. In time-based evaluation, test-set geohashes not seen in training will be imputed with the median.


In [31]:
geo_stats = train.groupby("geohash")["demand"].agg(
    ["mean", "std", "max", "min"]
).reset_index()

geo_stats.columns = [
    "geohash",
    "geo_mean",
    "geo_std",
    "geo_max",
    "geo_min"
]

train = train.merge(
    geo_stats,
    on="geohash",
    how="left"
)

test = test.merge(
    geo_stats,
    on="geohash",
    how="left"
)

In [32]:
for col in [
    "geo_mean",
    "geo_std",
    "geo_max",
    "geo_min"
]:
    train[col] = train[col].fillna(
        train[col].median()
    )

    test[col] = test[col].fillna(
        train[col].median()
    )

### 3.2 Cyclical Temporal Features

A raw hour integer is not naturally cyclical — the model would treat hour 23 and hour 0 as far apart. Encoding `hour` as **sine and cosine projections** preserves the circular structure of the 24-hour clock:

$$
\text{sin\_hour} = \sin\!\left(\frac{2\pi \cdot \text{hour}}{24}\right), \quad
\text{cos\_hour} = \cos\!\left(\frac{2\pi \cdot \text{hour}}{24}\right)
$$


In [33]:
train["hour"] = train["time_minutes"] // 60
test["hour"] = test["time_minutes"] // 60

train["sin_hour"] = np.sin(
    2 * np.pi * train["hour"] / 24
)

train["cos_hour"] = np.cos(
    2 * np.pi * train["hour"] / 24
)

test["sin_hour"] = np.sin(
    2 * np.pi * test["hour"] / 24
)

test["cos_hour"] = np.cos(
    2 * np.pi * test["hour"] / 24
)

### 3.3 Baseline Feature Set & Target Definition

Define the **16-feature baseline set** (`features`) and target column (`demand`).

Feature groups:

| Group | Features |
|-------|----------|
| Spatial | `geohash` |
| Temporal | `day`, `time_minutes`, `hour`, `sin_hour`, `cos_hour` |
| Road infrastructure | `RoadType`, `NumberofLanes`, `LargeVehicles`, `Landmarks` |
| Environmental | `Temperature`, `Weather` |
| Geohash aggregates | `geo_mean`, `geo_std`, `geo_max`, `geo_min` |


In [34]:
features = [
    "geohash",

    "day",
    "time_minutes",
    "hour",
    "sin_hour",
    "cos_hour",

    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "Landmarks",
    "Temperature",
    "Weather",

    "geo_mean",
    "geo_std",
    "geo_max",
    "geo_min"
]

target = "demand"

---

## 4. Model V1 — Baseline CatBoost (Random Split)

An initial CatBoost model is trained on an 80/20 random split to get a quick baseline RMSE before adopting a more rigorous time-based evaluation scheme.

### 4.1 Train / Validation Split


In [35]:
X_train, X_val, y_train, y_val = train_test_split(
    train[features],
    train[target],
    test_size=0.2,
    random_state=42
)

### 4.2 Categorical Feature Declaration

CatBoost handles categorical features natively — no one-hot encoding needed. The five categorical columns are declared here.


In [36]:
cat_features = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

### 4.3 Model Training

**Hyperparameters (V1):**

| Parameter | Value |
|-----------|-------|
| `iterations` | 1000 |
| `depth` | 8 |
| `learning_rate` | 0.05 |
| `loss_function` | RMSE |


In [37]:
model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val)
)

0:	learn: 0.1362150	test: 0.1362297	best: 0.1362297 (0)	total: 49.9ms	remaining: 49.8s
100:	learn: 0.0369318	test: 0.0378255	best: 0.0378255 (100)	total: 7.65s	remaining: 1m 8s
200:	learn: 0.0336905	test: 0.0350155	best: 0.0350155 (200)	total: 21.5s	remaining: 1m 25s
300:	learn: 0.0319375	test: 0.0337740	best: 0.0337735 (299)	total: 39.5s	remaining: 1m 31s
400:	learn: 0.0306514	test: 0.0329258	best: 0.0329258 (400)	total: 57.7s	remaining: 1m 26s
500:	learn: 0.0296822	test: 0.0323076	best: 0.0323076 (500)	total: 1m 15s	remaining: 1m 15s
600:	learn: 0.0289789	test: 0.0318977	best: 0.0318977 (600)	total: 1m 29s	remaining: 59.6s
700:	learn: 0.0282960	test: 0.0315157	best: 0.0315157 (700)	total: 1m 46s	remaining: 45.4s
800:	learn: 0.0278077	test: 0.0312857	best: 0.0312857 (800)	total: 2m 4s	remaining: 30.8s
900:	learn: 0.0273135	test: 0.0310509	best: 0.0310509 (900)	total: 2m 22s	remaining: 15.7s
999:	learn: 0.0268517	test: 0.0308202	best: 0.0308202 (999)	total: 2m 40s	remaining: 0us

bestT

CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

### 4.4 Evaluation — Validation RMSE


In [38]:
pred = model.predict(X_val)

mse = mean_squared_error(
    y_val,
    pred
)

rmse = np.sqrt(mse)

print("RMSE:", rmse)

RMSE: 0.03082024511120351


### 4.5 Prediction Distribution

Inspect the raw prediction range to check for out-of-bound values (demand should lie in [0, 1]).


In [39]:
print("Min :", pred.min())
print("Max :", pred.max())
print("Mean:", pred.mean())

Min : -0.00557015213518007
Max : 1.0576010335517902
Mean: 0.0944611018761955


### 4.6 Feature Importance

CatBoost outputs percentage-based feature importances. The top drivers are `RoadType` and `geo_mean`, confirming that both road category and historical zone demand are the strongest signals.


In [40]:
importance = model.get_feature_importance(
    prettified=True
)

print(importance)

       Feature Id  Importances
0        RoadType    28.372240
1        geo_mean    21.519525
2         geo_std     8.404855
3   NumberofLanes     6.906197
4         geo_max     6.000958
5        sin_hour     4.864417
6   LargeVehicles     4.711130
7    time_minutes     4.541937
8        cos_hour     4.225860
9         geo_min     3.171310
10           hour     2.872863
11        geohash     2.198553
12            day     1.564504
13    Temperature     0.393098
14        Weather     0.162909
15      Landmarks     0.089645


### 4.7 Full-Data Retraining & Submission (V1)

Retrain on the **entire training set** (no hold-out) to maximise signal before generating the test-set predictions.


In [41]:
final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

final_model.fit(
    train[features],
    train[target],
    cat_features=cat_features
)

0:	learn: 0.1360306	total: 67.2ms	remaining: 1m 7s
100:	learn: 0.0368038	total: 8.86s	remaining: 1m 18s
200:	learn: 0.0336991	total: 25.6s	remaining: 1m 41s
300:	learn: 0.0317423	total: 34.7s	remaining: 1m 20s
400:	learn: 0.0304642	total: 46.4s	remaining: 1m 9s
500:	learn: 0.0296111	total: 54.4s	remaining: 54.2s
600:	learn: 0.0288168	total: 1m 2s	remaining: 41.6s
700:	learn: 0.0282146	total: 1m 18s	remaining: 33.6s
800:	learn: 0.0277191	total: 1m 37s	remaining: 24.2s
900:	learn: 0.0272652	total: 1m 55s	remaining: 12.7s
999:	learn: 0.0268432	total: 2m 13s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

### 4.8 Test Predictions — Post-processing

Clip predictions to [0, 1] to satisfy the competition's demand range constraint.


In [42]:
test_pred = final_model.predict(
    test[features]
)

test_pred = np.clip(
    test_pred,
    0,
    1
)

### 4.9 Create & Export Submission (V1)


In [43]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_pred
})

submission.to_csv(
    "../outputs/submission_v1.csv",
    index=False
)

submission.head()

,Index,demand
0,0,0.047839
1,1,0.034052
2,2,0.026700
3,3,0.037781
4,4,0.055902


In [44]:
print(submission.shape)
print(submission.isnull().sum())

(41778, 2)
Index     0
demand    0
dtype: int64


### 4.10 Feature Importance — Full Model


In [45]:
importance = model.get_feature_importance(
    prettified=True
)

importance.head(20)

,Feature Id,Importances
0,RoadType,28.372240
1,geo_mean,21.519525
2,geo_std,8.404855
3,NumberofLanes,6.906197
4,geo_max,6.000958
5,sin_hour,4.864417
6,LargeVehicles,4.711130
7,time_minutes,4.541937
8,cos_hour,4.225860
9,geo_min,3.171310


---

## 5. Temporal Analysis

Before adopting a time-based split, we examine how demand varies across days and timestamps.  
This helps understand whether Day 49 is structurally different from Day 48 — a key question for the leakage-aware V2 features.

### 5.1 Mean Demand by Day

Day 49 shows ~14% higher mean demand than Day 48, suggesting a genuine distributional shift between the two days.


In [46]:
print(train.groupby("day")["demand"].mean())

day
48    0.092659
49    0.105262
Name: demand, dtype: float64


### 5.2 Day 49 — Available Timestamps

Day 49 only has **9 early-morning timestamps** (midnight to ~2:00 AM) in the training data.  
This is the signal leveraged in Model V2 to enrich geohash-level features.


In [47]:
train_day49 = train[train["day"] == 49]

print("Unique timestamps:",
      train_day49["timestamp"].nunique())

print(
    sorted(
        train_day49["timestamp"].unique()
    )
)

Unique timestamps: 9
['0:0', '0:15', '0:30', '0:45', '1:0', '1:15', '1:30', '1:45', '2:0']


### 5.3 Mean Demand by Timestamp (All Days)

Demand follows a typical urban diurnal pattern: low overnight, rising through morning, peaking mid-day.


In [48]:
train.groupby("timestamp")["demand"].mean().sort_index()

timestamp
0:0     0.081056
0:15    0.081929
0:30    0.084357
0:45    0.085994
10:0    0.110319
          ...   
8:45    0.105793
9:0     0.107004
9:15    0.112049
9:30    0.109178
9:45    0.108777
Name: demand, Length: 96, dtype: float64

---

## 6. Time-Based Validation

Random splits over-estimate real-world performance because they allow the model to "see" future data. A proper evaluation splits on time: **train on Day 48, validate on Day 49**.

| Split | Day | Rows |
|-------|-----|------|
| Train | 48 | 69,427 |
| Validation | 49 | 7,872 |

### 6.1 Temporal Split


In [49]:
train_time = train[train["day"] == 48].copy()
valid_time = train[train["day"] == 49].copy()

print(train_time.shape)
print(valid_time.shape)

(69427, 19)
(7872, 19)


In [50]:
X_train_time = train_time[features]
y_train_time = train_time[target]

X_valid_time = valid_time[features]
y_valid_time = valid_time[target]

### 6.2 Model V1 on Time-Based Validation

Training on Day 48 and validating on Day 49 reveals that the baseline RMSE rises from **0.031** (random split) to **0.065** — a more realistic picture of generalisation.


In [51]:
time_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

time_model.fit(
    X_train_time,
    y_train_time,
    cat_features=cat_features,
    eval_set=(X_valid_time, y_valid_time)
)

0:	learn: 0.1356583	test: 0.1394356	best: 0.1394356 (0)	total: 66.5ms	remaining: 1m 6s
100:	learn: 0.0351517	test: 0.0647607	best: 0.0646841 (86)	total: 12.8s	remaining: 1m 54s
200:	learn: 0.0322879	test: 0.0667623	best: 0.0646841 (86)	total: 32s	remaining: 2m 7s
300:	learn: 0.0303744	test: 0.0681097	best: 0.0646841 (86)	total: 51.1s	remaining: 1m 58s
400:	learn: 0.0292052	test: 0.0689600	best: 0.0646841 (86)	total: 1m 10s	remaining: 1m 44s
500:	learn: 0.0283299	test: 0.0694452	best: 0.0646841 (86)	total: 1m 29s	remaining: 1m 28s
600:	learn: 0.0276787	test: 0.0697960	best: 0.0646841 (86)	total: 1m 47s	remaining: 1m 11s
700:	learn: 0.0271364	test: 0.0701112	best: 0.0646841 (86)	total: 2m 6s	remaining: 54.1s
800:	learn: 0.0266496	test: 0.0703321	best: 0.0646841 (86)	total: 2m 25s	remaining: 36.3s
900:	learn: 0.0262156	test: 0.0706076	best: 0.0646841 (86)	total: 2m 45s	remaining: 18.2s
999:	learn: 0.0258611	test: 0.0707678	best: 0.0646841 (86)	total: 3m 4s	remaining: 0us

bestTest = 0.064

CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [52]:
pred_time = time_model.predict(X_valid_time)

rmse_time = np.sqrt(
    mean_squared_error(
        y_valid_time,
        pred_time
    )
)

print("Time RMSE:", rmse_time)

Time RMSE: 0.0646841209227351


---

## 7. Model V2 — Early Day 49 Geohash Features

**Motivation:** The test set covers Day 49. The training data contains a few early-morning observations from Day 49 (timestamps 0:00 – 2:00). These can be aggregated into zone-level statistics (`early49_mean`, `early49_std`, `early49_max`) that act as a **same-day signal** for the model.

> This is not data leakage because the early timestamps precede the timestamps we are predicting.

### 7.1 Extracting Early Day 49 Signals


In [53]:
day49_early = train[
    train["day"] == 49
].copy()

early_stats = day49_early.groupby(
    "geohash"
)["demand"].agg(
    ["mean", "std", "max"]
).reset_index()

early_stats.columns = [
    "geohash",
    "early49_mean",
    "early49_std",
    "early49_max"
]

early_stats.head()

,geohash,early49_mean,early49_std,early49_max
0,qp02z1,0.043114,0.024403,0.071038
1,qp02z3,0.030260,NaN,0.030260
2,qp02z6,0.004002,0.000369,0.004263
3,qp02z7,0.001361,NaN,0.001361
4,qp02z9,0.014499,0.008734,0.025590


### 7.2 Merging Features & Imputation


In [54]:
train_v2 = train.merge(
    early_stats,
    on="geohash",
    how="left"
)

test_v2 = test.merge(
    early_stats,
    on="geohash",
    how="left"
)

print(train_v2.shape)
print(test_v2.shape)

(77299, 22)
(41778, 21)


In [55]:
for col in [
    "early49_mean",
    "early49_std",
    "early49_max"
]:
    train_v2[col] = train_v2[col].fillna(
        train_v2[col].median()
    )

    test_v2[col] = test_v2[col].fillna(
        train_v2[col].median()
    )

print(
    train_v2[
        ["early49_mean",
         "early49_std",
         "early49_max"]
    ].isnull().sum()
)

early49_mean    0
early49_std     0
early49_max     0
dtype: int64


### 7.3 Extended Feature Set (V2)

Three new features are appended — `early49_mean`, `early49_std`, `early49_max` — bringing the total to **19 features**.


In [56]:
features_v2 = features + [
    "early49_mean",
    "early49_std",
    "early49_max"
]

print(len(features_v2))

19


### 7.4 Time-Based Split for V2 Data


In [57]:
train_time_v2 = train_v2[
    train_v2["day"] == 48
].copy()

valid_time_v2 = train_v2[
    train_v2["day"] == 49
].copy()

print(train_time_v2.shape)
print(valid_time_v2.shape)

(69427, 22)
(7872, 22)


In [58]:
X_train_v2 = train_time_v2[
    features_v2
]

y_train_v2 = train_time_v2[
    target
]

X_valid_v2 = valid_time_v2[
    features_v2
]

y_valid_v2 = valid_time_v2[
    target
]

### 7.5 Model Training (V2)


In [59]:
model_v2 = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

model_v2.fit(
    X_train_v2,
    y_train_v2,
    cat_features=cat_features,
    eval_set=(
        X_valid_v2,
        y_valid_v2
    )
)

0:	learn: 0.1356435	test: 0.1403709	best: 0.1403709 (0)	total: 70.1ms	remaining: 1m 9s
100:	learn: 0.0343233	test: 0.0641588	best: 0.0637514 (76)	total: 12s	remaining: 1m 47s
200:	learn: 0.0310893	test: 0.0665947	best: 0.0637514 (76)	total: 30.7s	remaining: 2m 1s
300:	learn: 0.0292482	test: 0.0681459	best: 0.0637514 (76)	total: 49.6s	remaining: 1m 55s
400:	learn: 0.0281878	test: 0.0691626	best: 0.0637514 (76)	total: 1m 8s	remaining: 1m 42s
500:	learn: 0.0274226	test: 0.0698234	best: 0.0637514 (76)	total: 1m 27s	remaining: 1m 27s
600:	learn: 0.0268171	test: 0.0702869	best: 0.0637514 (76)	total: 1m 46s	remaining: 1m 10s
700:	learn: 0.0262546	test: 0.0706820	best: 0.0637514 (76)	total: 1m 59s	remaining: 51s
800:	learn: 0.0257552	test: 0.0710197	best: 0.0637514 (76)	total: 2m 19s	remaining: 34.6s
900:	learn: 0.0252975	test: 0.0712812	best: 0.0637514 (76)	total: 2m 31s	remaining: 16.6s
999:	learn: 0.0249019	test: 0.0715392	best: 0.0637514 (76)	total: 2m 40s	remaining: 0us

bestTest = 0.0637

CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

### 7.6 V1 vs V2 — RMSE Comparison

Adding early Day-49 signals reduces time-based RMSE from **0.06468 → 0.06375**, a meaningful improvement.


In [60]:
pred_v2 = model_v2.predict(
    X_valid_v2
)

rmse_v2 = np.sqrt(
    mean_squared_error(
        y_valid_v2,
        pred_v2
    )
)

print("V1 Time RMSE :", rmse_time)
print("V2 Time RMSE :", rmse_v2)

V1 Time RMSE : 0.0646841209227351
V2 Time RMSE : 0.06375138289822031


### 7.7 Feature Importance (V2)

`RoadType` and `geo_mean` remain dominant. The new `early49_mean` feature enters the top-15, confirming the signal is being leveraged.


In [61]:
importance_v2 = model_v2.get_feature_importance(
    prettified=True
)

importance_v2.head(20)

,Feature Id,Importances
0,RoadType,34.131587
1,geo_mean,24.928555
2,NumberofLanes,9.078054
3,geo_std,5.547451
4,time_minutes,4.746540
5,cos_hour,4.535755
6,sin_hour,4.415457
7,LargeVehicles,2.841268
8,geo_max,2.701740
9,hour,2.460701


---

## 8. Model V3 — Geohash × Timestamp Interaction Features

**Motivation:** Traffic demand is highly dependent on both *where* (geohash) and *when* (timestamp). A new feature `geo_time_mean` captures the **historical average demand for each (geohash, timestamp) pair**, encoding fine-grained spatio-temporal patterns.

### 8.1 Computing Geohash × Timestamp Statistics


In [62]:
geo_time_stats = train.groupby(
    ["geohash", "timestamp"]
)["demand"].mean().reset_index()

geo_time_stats.columns = [
    "geohash",
    "timestamp",
    "geo_time_mean"
]

geo_time_stats.head()

,geohash,timestamp,geo_time_mean
0,qp02yc,10:30,0.046790
1,qp02yc,10:45,0.021158
2,qp02yc,1:0,0.005397
3,qp02yc,2:30,0.012944
4,qp02yc,2:45,0.025961


### 8.2 Merging & Imputing


In [63]:
train_v3 = train_v2.merge(
    geo_time_stats,
    on=["geohash", "timestamp"],
    how="left"
)

test_v3 = test_v2.merge(
    geo_time_stats,
    on=["geohash", "timestamp"],
    how="left"
)

In [64]:
train_v3["geo_time_mean"] = train_v3[
    "geo_time_mean"
].fillna(
    train_v3["geo_time_mean"].median()
)

test_v3["geo_time_mean"] = test_v3[
    "geo_time_mean"
].fillna(
    train_v3["geo_time_mean"].median()
)

### 8.3 Extended Feature Set (V3)


In [65]:
features_v3 = features_v2 + [
    "geo_time_mean"
]

### 8.4 Time-Based Split for V3 Data


In [66]:
train_time_v3 = train_v3[
    train_v3["day"] == 48
]

valid_time_v3 = train_v3[
    train_v3["day"] == 49
]

In [67]:
X_train_v3 = train_time_v3[
    features_v3
]

y_train_v3 = train_time_v3[
    target
]

X_valid_v3 = valid_time_v3[
    features_v3
]

y_valid_v3 = valid_time_v3[
    target
]

### 8.5 Model Training (V3)


In [68]:
model_v3 = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

model_v3.fit(
    X_train_v3,
    y_train_v3,
    cat_features=cat_features,
    eval_set=(
        X_valid_v3,
        y_valid_v3
    )
)

0:	learn: 0.1352441	test: 0.1404084	best: 0.1404084 (0)	total: 67.3ms	remaining: 1m 7s
100:	learn: 0.0093460	test: 0.0666349	best: 0.0661336 (74)	total: 7.72s	remaining: 1m 8s
200:	learn: 0.0079260	test: 0.0684442	best: 0.0661336 (74)	total: 17s	remaining: 1m 7s
300:	learn: 0.0073228	test: 0.0692424	best: 0.0661336 (74)	total: 25.6s	remaining: 59.4s
400:	learn: 0.0069250	test: 0.0697588	best: 0.0661336 (74)	total: 34s	remaining: 50.8s
500:	learn: 0.0065603	test: 0.0703515	best: 0.0661336 (74)	total: 43.2s	remaining: 43s
600:	learn: 0.0062614	test: 0.0708268	best: 0.0661336 (74)	total: 1m 1s	remaining: 40.9s
700:	learn: 0.0060220	test: 0.0711771	best: 0.0661336 (74)	total: 1m 20s	remaining: 34.3s
800:	learn: 0.0057964	test: 0.0715017	best: 0.0661336 (74)	total: 1m 39s	remaining: 24.7s
900:	learn: 0.0056041	test: 0.0717754	best: 0.0661336 (74)	total: 1m 47s	remaining: 11.8s
999:	learn: 0.0054244	test: 0.0720073	best: 0.0661336 (74)	total: 2m 5s	remaining: 0us

bestTest = 0.06613361359
be

CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

### 8.6 V1 / V2 / V3 — RMSE Comparison

| Model | Features | Time RMSE |
|-------|----------|-----------|
| V1 | 16 baseline | 0.06468 |
| V2 | + early Day-49 (19) | **0.06375** ✅ |
| V3 | + geo×time (20) | 0.06613 |

V3 does **not** improve on V2 — the `geo_time_mean` feature likely causes overfitting given the small Day-49 training window. V2 is retained as the best performing feature set.


In [69]:
pred_v3 = model_v3.predict(
    X_valid_v3
)

rmse_v3 = np.sqrt(
    mean_squared_error(
        y_valid_v3,
        pred_v3
    )
)

print("V1 :", rmse_time)
print("V2 :", rmse_v2)
print("V3 :", rmse_v3)

V1 : 0.0646841209227351
V2 : 0.06375138289822031
V3 : 0.06613361336623313


---

## 9. Exploratory Analysis — Road Infrastructure

Investigate how `RoadType`, `LargeVehicles`, and `NumberofLanes` relate to demand. These findings inform feature engineering decisions in later iterations.

### 9.1 RoadType Distribution


In [70]:
print(train["RoadType"].value_counts(dropna=False))

RoadType
Residential    69230
Street          3909
Highway         3560
Missing          600
Name: count, dtype: int64


### 9.2 Demand Statistics by Road Type

Highways carry dramatically higher mean demand (~0.61) than Residential roads (~0.057), confirming that `RoadType` is the most important feature.


In [71]:
print(train.groupby("RoadType")["demand"].describe())

               count      mean       std           min       25%       50%  \
RoadType                                                                     
Highway       3560.0  0.610756  0.229419  3.500089e-01  0.413629  0.526432   
Missing        600.0  0.098263  0.146106  1.965464e-04  0.017822  0.048059   
Residential  69230.0  0.057209  0.052057  6.245650e-07  0.016203  0.040517   
Street        3909.0  0.273164  0.036693  2.200159e-01  0.240702  0.268124   

                  75%       max  
RoadType                         
Highway      0.790095  1.000000  
Missing      0.111594  1.000000  
Residential  0.084468  0.219997  
Street       0.301890  0.349908  


### 9.3 Demand by Road Type × Large Vehicles


In [72]:
print(
    train.groupby(["RoadType", "LargeVehicles"])["demand"]
    .mean()
)

RoadType     LargeVehicles
Highway      Allowed          0.610756
Missing      Allowed          0.147438
             Not Allowed      0.074406
Residential  Allowed          0.057253
             Not Allowed      0.057188
Street       Not Allowed      0.273164
Name: demand, dtype: float64


### 9.4 Demand by Number of Lanes

Roads with 4–5 lanes exhibit very high demand (~0.60), aligning closely with the Highway category. Single-lane roads show the lowest demand.


In [73]:
print(
    train.groupby("NumberofLanes")["demand"]
    .describe()
)

                 count      mean       std           min       25%       50%  \
NumberofLanes                                                                  
1              27411.0  0.088104  0.090681  3.896741e-06  0.019475  0.052204   
2              24127.0  0.077488  0.124543  1.408894e-06  0.016800  0.043005   
3              23919.0  0.077859  0.125011  6.245650e-07  0.017063  0.042845   
4                926.0  0.602882  0.225795  3.502258e-01  0.412532  0.513090   
5                916.0  0.607556  0.226946  3.500089e-01  0.413233  0.521678   

                    75%       max  
NumberofLanes                      
1              0.126136  0.349908  
2              0.092359  1.000000  
3              0.092126  1.000000  
4              0.780396  1.000000  
5              0.779799  1.000000  


---

## 10. LightGBM Exploration & Ensemble

A LightGBM model is trained on the same V2 feature set as a second learner. The goal is to explore whether a CatBoost + LightGBM ensemble can reduce variance and improve RMSE.

### 10.1 Library Import


In [74]:
from lightgbm import LGBMRegressor

### 10.2 Data Preparation — Categorical Encoding

LightGBM requires categorical columns to be cast to `pandas.Categorical` dtype (it cannot ingest raw string columns as CatBoost can).


In [80]:
train_lgb = train_time_v2.copy()
valid_lgb = valid_time_v2.copy()

cat_cols = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

for col in cat_cols:
    train_lgb[col] = train_lgb[col].astype("category")
    valid_lgb[col] = valid_lgb[col].astype("category")

### 10.3 Model Training

**LightGBM Hyperparameters:**

| Parameter | Value |
|-----------|-------|
| `n_estimators` | 1000 |
| `learning_rate` | 0.05 |
| `num_leaves` | 63 |
| `random_state` | 42 |


In [81]:
lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42
)

lgb_model.fit(
    train_lgb[features_v2],
    train_lgb[target]
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000673 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3279
[LightGBM] [Info] Number of data points in the train set: 69427, number of used features: 18
[LightGBM] [Info] Start training from score 0.092659


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### 10.4 LightGBM Evaluation


In [82]:
pred_lgb = lgb_model.predict(
    valid_lgb[features_v2]
)

rmse_lgb = np.sqrt(
    mean_squared_error(
        valid_lgb[target],
        pred_lgb
    )
)

print("LightGBM RMSE:", rmse_lgb)

LightGBM RMSE: 0.07247361198959502


### 10.5 Ensemble: CatBoost 70% + LightGBM 30%

The ensemble (weighted average) yields RMSE **0.06534**, which is between V2 CatBoost (0.06375) and LightGBM (0.07247). CatBoost dominates on this dataset; the ensemble does not outperform the best solo model.


In [83]:
ensemble_pred = (
    pred_v2 * 0.7
    +
    pred_lgb * 0.3
)

ensemble_rmse = np.sqrt(
    mean_squared_error(
        valid_lgb[target],
        ensemble_pred
    )
)

print("Ensemble RMSE:", ensemble_rmse)

Ensemble RMSE: 0.06534098714242637


### 10.6 Data Shape Verification


In [84]:
print(train_lgb.shape)
print(valid_lgb.shape)

(69427, 22)
(7872, 22)


---

## 11. Model V2.1 — CatBoost Hyperparameter Search

A lightweight grid search over `depth` and `learning_rate` is performed to find a better configuration than the V2 defaults. Each configuration uses `early_stopping_rounds=100` with up to 3000 iterations.

### 11.1 Evaluation Helper Function


In [86]:
def evaluate_catboost(
    depth,
    learning_rate,
    iterations=3000
):

    model = CatBoostRegressor(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        loss_function="RMSE",
        verbose=False
    )

    model.fit(
        X_train_v2,
        y_train_v2,
        cat_features=cat_features,
        eval_set=(X_valid_v2, y_valid_v2),
        early_stopping_rounds=100
    )

    pred = model.predict(X_valid_v2)

    rmse = np.sqrt(
        mean_squared_error(
            y_valid_v2,
            pred
        )
    )

    return rmse, model

### 11.2 Grid Search — Configurations

Four configurations are evaluated:

| Config | Depth | LR | Notes |
|--------|-------|----|-------|
| A | 8 | 0.05 | V2 baseline |
| B | 6 | 0.03 | Shallower, slower LR |
| C | 10 | 0.03 | Deeper, slower LR |
| D | 7 | 0.07 | **Winner** — best RMSE |


In [87]:
configs = [
    ("A", 8, 0.05),  # current champion
    ("B", 6, 0.03),
    ("C", 10, 0.03),
    ("D", 7, 0.07),
]

results = []

for name, depth, lr in configs:

    rmse, model = evaluate_catboost(
        depth=depth,
        learning_rate=lr
    )

    results.append([
        name,
        depth,
        lr,
        rmse
    ])

    print(
        f"{name} | depth={depth} | "
        f"lr={lr} | RMSE={rmse:.6f}"
    )

A | depth=8 | lr=0.05 | RMSE=0.063751
B | depth=6 | lr=0.03 | RMSE=0.062146
C | depth=10 | lr=0.03 | RMSE=0.066097
D | depth=7 | lr=0.07 | RMSE=0.061971


### 11.3 Results Summary


In [88]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Depth",
        "LearningRate",
        "RMSE"
    ]
)

results_df.sort_values(
    "RMSE"
)

,Model,Depth,LearningRate,RMSE
3,D,7,0.07,0.061971
1,B,6,0.03,0.062146
0,A,8,0.05,0.063751
2,C,10,0.03,0.066097


---

## 12. Final Submission Model — CatBoost Config D

The best configuration from the hyperparameter search (**Config D**: `depth=7`, `lr=0.07`, `iterations=3000`) is trained on the **full V2 training set** (all days) to maximise data utilisation before generating the final test predictions.

### 12.1 Model Instantiation


In [89]:
final_model = CatBoostRegressor(
    iterations=3000,
    depth=7,
    learning_rate=0.07,
    loss_function="RMSE",
    verbose=100
)

### 12.2 Training on Full Dataset


In [90]:
final_model.fit(
    train_v2[features_v2],
    train_v2[target],
    cat_features=cat_features
)

0:	learn: 0.1337179	total: 151ms	remaining: 7m 31s
100:	learn: 0.0350266	total: 19.8s	remaining: 9m 26s
200:	learn: 0.0316915	total: 40s	remaining: 9m 16s
300:	learn: 0.0298700	total: 1m	remaining: 9m 1s
400:	learn: 0.0286976	total: 1m 20s	remaining: 8m 41s
500:	learn: 0.0279304	total: 1m 32s	remaining: 7m 40s
600:	learn: 0.0273157	total: 1m 51s	remaining: 7m 25s
700:	learn: 0.0268158	total: 2m 11s	remaining: 7m 11s
800:	learn: 0.0263366	total: 2m 31s	remaining: 6m 54s
900:	learn: 0.0259298	total: 2m 51s	remaining: 6m 39s
1000:	learn: 0.0255782	total: 3m 12s	remaining: 6m 23s
1100:	learn: 0.0252444	total: 3m 32s	remaining: 6m 6s
1200:	learn: 0.0249519	total: 3m 52s	remaining: 5m 47s
1300:	learn: 0.0246515	total: 4m 12s	remaining: 5m 29s
1400:	learn: 0.0243912	total: 4m 32s	remaining: 5m 11s
1500:	learn: 0.0241296	total: 4m 52s	remaining: 4m 52s
1600:	learn: 0.0239243	total: 5m 8s	remaining: 4m 29s
1700:	learn: 0.0237325	total: 5m 21s	remaining: 4m 5s
1800:	learn: 0.0235296	total: 5m 41

CatBoostRegressor(depth=7, iterations=3000, learning_rate=0.07, loss_function='RMSE', verbose=100)

### 12.3 Raw Predictions & Post-processing

Raw predictions from tree models can fall outside [0, 1]. Inspect the range and clip.


In [91]:
pred_test = final_model.predict(
    test_v2[features_v2]
)

print(pred_test.min())
print(pred_test.max())
print(pred_test.mean())

-0.013605965291802774
1.152413470910528
0.12124040312907705


In [92]:
pred_test = np.clip(
    pred_test,
    0,
    1
)

print(pred_test.min())
print(pred_test.max())
print(pred_test.mean())

0.0
1.0
0.12109275164213261


### 12.4 Build Submission DataFrame


In [93]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": pred_test
})

submission.head()

,Index,demand
0,0,0.052855
1,1,0.016660
2,2,0.053739
3,3,0.002934
4,4,0.076776


### 12.5 Submission Validation — Shape & Null Check


In [94]:
print(submission.shape)

print(
    submission.isnull().sum()
)

(41778, 2)
Index     0
demand    0
dtype: int64


### 12.6 Export Submission File


In [95]:
submission.to_csv(
    "../outputs/submission_model_D.csv",
    index=False
)

### 12.7 Sanity Check — Random Sample


In [96]:
submission.sample(10)

,Index,demand
3607,3607,0.034482
27286,27286,0.006143
7860,7860,0.035358
19692,19692,0.062681
21266,21266,0.007531
12797,12797,0.017000
40964,40964,0.012846
6770,6770,0.065880
38065,38065,0.069468
24146,24146,0.049731


### 12.8 Export Model Comparison Table


In [97]:
results_df
results_df.to_csv(
    "../outputs/model_comparison.csv",
    index=False
)

---

## 13. Summary & Next Steps

### Results

| Model | Validation | RMSE |
|-------|-----------|------|
| V1 (random split) | Random 80/20 | 0.0308 |
| V1 (time split) | Day 48 → 49 | 0.0647 |
| V2 (early Day-49) | Day 48 → 49 | **0.0638** |
| V3 (geo×time) | Day 48 → 49 | 0.0661 |
| **Final (Config D)** | **Full train** | **submitted** |

### Possible Improvements

- **Rolling window features** — multi-day demand history per geohash
- **Lag features** — demand 1-step and 4-steps (1 hour) prior  
- **Richer hyperparameter search** — Optuna / Bayesian optimisation  
- **Stacking** — use a meta-learner on top of CatBoost + LightGBM + XGBoost  
- **Geo-spatial features** — distance to city centre, nearest landmark  
